# Chapter 23 — Comprehensive Statistical Analysis

## Final Project: Learning Program Analytics

This chapter brings together the statistical workflow developed throughout Chapters 01–22.

**Main question**

> What factors are associated with learning outcomes and course completion?

The goal is not to use every statistical method simply because it exists. The goal is to choose methods that match the research question, variable types, assumptions, and interpretation needs.

## 1. Project Overview

A statistical analysis is a process:

$$
\text{Question}
\rightarrow
\text{Data}
\rightarrow
\text{Exploration}
\rightarrow
\text{Inference}
\rightarrow
\text{Modeling}
\rightarrow
\text{Interpretation}
$$

We will analyze a synthetic dataset containing 240 participants in a fictional learning program.

## 2. Research Questions

- **RQ1:** What are the main characteristics of the participants?
- **RQ2:** Does final performance differ across learning styles?
- **RQ3:** Are study behavior and learning outcomes associated?
- **RQ4:** Which factors are associated with final performance when considered together?
- **RQ5:** Which factors are associated with course completion?
- **RQ6:** How uncertain are our key statistical estimates?

## 3. Load the Dataset

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.outliers_influence import variance_inflation_factor

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root containing a data/ directory.")

ROOT = find_project_root(Path.cwd())
DATA_PATH = ROOT / "data" / "23_comprehensive_statistical_analysis.csv"

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## 4. Understand the Variables

| Variable | Type | Role |
|---|---|---|
| `participant_id` | Identifier | Record identifier |
| `program_track` | Categorical | Group / explanatory |
| `study_style` | Categorical | Group / explanatory |
| `prior_score` | Quantitative | Explanatory |
| `study_hours_weekly` | Quantitative | Explanatory |
| `attendance_rate` | Quantitative | Explanatory |
| `support_sessions` | Discrete quantitative | Explanatory |
| `satisfaction_level` | Ordinal | Explanatory / descriptive |
| `final_score` | Quantitative | Main performance outcome |
| `completed` | Binary | Completion outcome |

## 5. Data Quality Check

In [ ]:
quality_check = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique()
})
quality_check

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

range_checks = {
    "prior_score_0_100": df["prior_score"].between(0, 100).all(),
    "study_hours_nonnegative": (df["study_hours_weekly"] >= 0).all(),
    "attendance_0_100": df["attendance_rate"].between(0, 100).all(),
    "satisfaction_1_5": df["satisfaction_level"].between(1, 5).all(),
    "final_score_0_100": df["final_score"].between(0, 100).all(),
    "completed_binary": df["completed"].isin([0, 1]).all(),
}
range_checks

# Part A — Exploratory Data Analysis

## 6. Descriptive Statistics

In [ ]:
numeric_cols = [
    "prior_score", "study_hours_weekly", "attendance_rate",
    "support_sessions", "satisfaction_level", "final_score"
]
df[numeric_cols].describe().T

In [ ]:
for col in ["program_track", "study_style", "satisfaction_level", "completed"]:
    print(f"\n{col}")
    print(df[col].value_counts().sort_index())

## 7. Distribution Analysis

In [ ]:
fig, ax = plt.subplots()
ax.hist(df["final_score"], bins=15, edgecolor="black")
ax.set_xlabel("Final score")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Final Scores")
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.boxplot(df["study_hours_weekly"], vert=True)
ax.set_ylabel("Study hours per week")
ax.set_title("Weekly Study Hours")
plt.show()

print("Mean:", df["study_hours_weekly"].mean())
print("Median:", df["study_hours_weekly"].median())
print("Skewness:", stats.skew(df["study_hours_weekly"], bias=False))

## 8. Explore Participant Groups

In [ ]:
df.groupby("study_style", observed=False).agg(
    participants=("participant_id", "count"),
    mean_final_score=("final_score", "mean"),
    median_final_score=("final_score", "median"),
    mean_attendance=("attendance_rate", "mean"),
    completion_rate=("completed", "mean")
).round(3)

## 9. Visual Comparison of Learning Outcomes

In [ ]:
styles = ["Self-paced", "Group", "Mentored"]
groups = [df.loc[df["study_style"] == s, "final_score"] for s in styles]

fig, ax = plt.subplots()
ax.boxplot(groups, tick_labels=styles)
ax.set_xlabel("Study style")
ax.set_ylabel("Final score")
ax.set_title("Final Score by Study Style")
plt.show()

# Part B — Statistical Inference

## 10. From Research Questions to Statistical Hypotheses

For the multi-group comparison:

$$
H_0:\mu_{\text{Self-paced}}=\mu_{\text{Group}}=\mu_{\text{Mentored}}
$$

$$
H_1:\text{At least one population mean differs}
$$

We use $\alpha=0.05$ as a conventional significance level, but interpretation will also consider effect size, confidence intervals, and practical importance.

## 11. Compare Learning Outcomes Between Two Groups

In [ ]:
two_group = df[df["study_style"].isin(["Self-paced", "Mentored"])]
self_paced = two_group.loc[two_group["study_style"] == "Self-paced", "final_score"]
mentored = two_group.loc[two_group["study_style"] == "Mentored", "final_score"]

welch = stats.ttest_ind(self_paced, mentored, equal_var=False)
mean_diff = mentored.mean() - self_paced.mean()

# Cohen's d using pooled SD as a descriptive standardized difference.
n1, n2 = len(self_paced), len(mentored)
s1, s2 = self_paced.std(ddof=1), mentored.std(ddof=1)
sp = np.sqrt(((n1 - 1)*s1**2 + (n2 - 1)*s2**2) / (n1 + n2 - 2))
cohens_d = mean_diff / sp

print("Welch t-test:", welch)
print("Mean difference (Mentored - Self-paced):", mean_diff)
print("Cohen's d:", cohens_d)

## 12. Compare Multiple Learning Styles

In [ ]:
anova = stats.f_oneway(*groups)
anova

In [ ]:
tukey = pairwise_tukeyhsd(
    endog=df["final_score"],
    groups=df["study_style"],
    alpha=0.05
)
print(tukey)

In [ ]:
grand_mean = df["final_score"].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
ss_total = ((df["final_score"] - grand_mean)**2).sum()
eta_squared = ss_between / ss_total
eta_squared

## 13. Association Between Categorical Variables

In [ ]:
contingency = pd.crosstab(df["study_style"], df["completed"])
contingency

In [ ]:
chi2, p, dof, expected = stats.chi2_contingency(contingency)
n = contingency.to_numpy().sum()
r, k = contingency.shape
cramers_v = np.sqrt(chi2 / (n * min(r - 1, k - 1)))

print("Chi-square:", chi2)
print("p-value:", p)
print("degrees of freedom:", dof)
print("Cramer's V:", cramers_v)
pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)

# Part C — Relationships Between Variables

## 14. Covariance and Correlation Analysis

In [ ]:
relationship_cols = [
    "prior_score", "study_hours_weekly", "attendance_rate",
    "support_sessions", "final_score"
]
df[relationship_cols].cov().round(2)

## 15. Correlation Matrix

In [ ]:
corr = df[relationship_cols].corr(method="pearson")
corr.round(3)

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(corr, vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.index)), corr.index)
ax.set_title("Pearson Correlation Matrix")
fig.colorbar(im, ax=ax)

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")

plt.tight_layout()
plt.show()

## 16. Pearson vs Spearman Correlation

In [ ]:
pearson = stats.pearsonr(df["study_hours_weekly"], df["final_score"])
spearman = stats.spearmanr(df["study_hours_weekly"], df["final_score"])

print("Pearson:", pearson)
print("Spearman:", spearman)

## 17. Correlation and Causation

A statistically significant correlation is evidence of association, not proof that one variable causes the other.

Possible explanations include:

- direct causal influence,
- reverse influence,
- confounding variables,
- selection mechanisms,
- or chance.

Because this dataset is analyzed as observational data, causal claims should not be made from correlation or regression alone.

# Part D — Modeling Final Performance

## 18. Simple Linear Regression

In [ ]:
simple_model = smf.ols(
    "final_score ~ study_hours_weekly",
    data=df
).fit()

simple_model.summary()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df["study_hours_weekly"], df["final_score"], alpha=0.65)

x = np.linspace(df["study_hours_weekly"].min(), df["study_hours_weekly"].max(), 100)
pred = simple_model.predict(pd.DataFrame({"study_hours_weekly": x}))
ax.plot(x, pred)

ax.set_xlabel("Study hours per week")
ax.set_ylabel("Final score")
ax.set_title("Simple Linear Regression")
plt.show()

## 19. Multiple Linear Regression

In [ ]:
multiple_model = smf.ols(
    "final_score ~ prior_score + study_hours_weekly + attendance_rate + "
    "support_sessions + C(program_track) + C(study_style)",
    data=df
).fit()

multiple_model.summary()

A coefficient in multiple regression describes the estimated association with the outcome **holding the other variables in the model constant**.

This conditional interpretation is different from a simple pairwise relationship.

## 20. Regression Diagnostics

In [ ]:
fitted = multiple_model.fittedvalues
resid = multiple_model.resid

fig, ax = plt.subplots()
ax.scatter(fitted, resid, alpha=0.65)
ax.axhline(0)
ax.set_xlabel("Fitted values")
ax.set_ylabel("Residuals")
ax.set_title("Residuals vs Fitted")
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.hist(resid, bins=15, edgecolor="black")
ax.set_xlabel("Residual")
ax.set_ylabel("Frequency")
ax.set_title("Residual Distribution")
plt.show()

In [ ]:
X_vif = df[["prior_score", "study_hours_weekly", "attendance_rate", "support_sessions"]].copy()
X_vif = sm.add_constant(X_vif)

vif = pd.DataFrame({
    "variable": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
vif

## 21. Interpret the Performance Model

In [ ]:
performance_results = pd.DataFrame({
    "coefficient": multiple_model.params,
    "p_value": multiple_model.pvalues,
    "ci_lower": multiple_model.conf_int()[0],
    "ci_upper": multiple_model.conf_int()[1]
})
performance_results.round(4)

Interpretation should combine:

- direction and magnitude of coefficients,
- uncertainty through confidence intervals,
- statistical evidence through p-values,
- model fit,
- diagnostics,
- and the practical meaning of the estimated differences.

A small p-value does not automatically imply a large or important effect.

# Part E — Modeling Course Completion

## 22. Explore the Binary Outcome

In [ ]:
completion_summary = df.groupby("completed").agg(
    participants=("participant_id", "count"),
    mean_final_score=("final_score", "mean"),
    mean_attendance=("attendance_rate", "mean"),
    mean_study_hours=("study_hours_weekly", "mean"),
    mean_support_sessions=("support_sessions", "mean")
)
completion_summary.round(2)

## 23. Logistic Regression

In [ ]:
logit_model = smf.logit(
    "completed ~ final_score + attendance_rate + study_hours_weekly + support_sessions",
    data=df
).fit(disp=False)

logit_model.summary()

## 24. Odds Ratios

In [ ]:
ci = logit_model.conf_int()
odds_ratios = pd.DataFrame({
    "odds_ratio": np.exp(logit_model.params),
    "ci_lower": np.exp(ci[0]),
    "ci_upper": np.exp(ci[1]),
    "p_value": logit_model.pvalues
})
odds_ratios.round(4)

For a one-unit increase in predictor $X$, the odds are multiplied by

$$
OR=e^{\beta_X}
$$

while the other variables in the model are held constant.

Odds ratios are not the same as changes in probability.

## 25. Predicted Probabilities

In [ ]:
df["predicted_completion_probability"] = logit_model.predict(df)

df[[
    "participant_id", "final_score", "attendance_rate",
    "study_hours_weekly", "completed",
    "predicted_completion_probability"
]].head(10)

In [ ]:
df["predicted_completed_050"] = (df["predicted_completion_probability"] >= 0.5).astype(int)
pd.crosstab(
    df["completed"],
    df["predicted_completed_050"],
    rownames=["Actual"],
    colnames=["Predicted"]
)

# Part F — Robustness and Uncertainty

## 26. Nonparametric Analysis

In [ ]:
mw = stats.mannwhitneyu(
    self_paced,
    mentored,
    alternative="two-sided"
)
mw

The Mann–Whitney U test is rank-based. Its broad null hypothesis concerns the distributions of the two groups.

It should not automatically be described as a “test of medians.” A location or median interpretation requires additional distributional conditions.

## 27. Bootstrap Analysis

In [ ]:
rng = np.random.default_rng(42)
B = 10_000

observed_mean = df["final_score"].mean()
bootstrap_means = np.empty(B)

values = df["final_score"].to_numpy()
n = len(values)

for b in range(B):
    sample = rng.choice(values, size=n, replace=True)
    bootstrap_means[b] = sample.mean()

bootstrap_se = bootstrap_means.std(ddof=1)

print("Observed mean:", observed_mean)
print("Bootstrap SE:", bootstrap_se)

In [ ]:
fig, ax = plt.subplots()
ax.hist(bootstrap_means, bins=40, edgecolor="black")
ax.axvline(observed_mean)
ax.set_xlabel("Bootstrap mean final score")
ax.set_ylabel("Frequency")
ax.set_title("Bootstrap Distribution of the Mean")
plt.show()

## 28. Bootstrap Confidence Interval

In [ ]:
bootstrap_ci = np.percentile(bootstrap_means, [2.5, 97.5])
bootstrap_ci

The percentile bootstrap interval uses the empirical quantiles of the bootstrap distribution:

$$
CI_{95\%}
=
[Q_{0.025}, Q_{0.975}]
$$

The bootstrap does not create information that was absent from the original sample. Its usefulness still depends on whether the observed sample is informative about the population of interest.

# Part G — Final Statistical Interpretation

## 29. Statistical Significance vs Practical Significance

Statistical significance asks whether the observed evidence is difficult to reconcile with a null model.

Practical significance asks whether the estimated effect is large enough to matter in context.

A complete interpretation should consider:

- p-values,
- effect sizes,
- regression coefficients,
- confidence intervals,
- uncertainty,
- and the practical context.

## 30. Answer the Research Questions

In [ ]:
print("RQ1 — Participant characteristics")
print(df[numeric_cols].describe().loc[["mean", "50%", "std"]].round(2))

print("\nRQ2 — Mean final score by study style")
print(df.groupby("study_style")["final_score"].mean().round(2))

print("\nRQ3 — Correlation with final score")
print(df[relationship_cols].corr()["final_score"].sort_values(ascending=False).round(3))

print("\nRQ4 — Multiple regression R-squared")
print("R-squared:", round(multiple_model.rsquared, 3))
print("Adjusted R-squared:", round(multiple_model.rsquared_adj, 3))

print("\nRQ5 — Completion rate")
print(round(df["completed"].mean(), 3))

print("\nRQ6 — Bootstrap 95% CI for mean final score")
print(np.round(bootstrap_ci, 2))

When writing the final answers, organize the interpretation around the **research questions**, not around the names of statistical procedures.

Instead of:

> “The ANOVA returned a p-value of ...”

prefer:

> “Final performance differed across learning styles, with ...”

and then provide the statistical evidence that supports the statement.

## 31. Limitations

This analysis has important limitations:

1. The dataset is **synthetic** and exists for educational purposes.
2. Relationships are analyzed as observational associations.
3. Important variables may be omitted from a real-world model.
4. Statistical estimates contain sampling uncertainty.
5. Regression models depend on assumptions and model specification.
6. Statistical association does not automatically establish causation.

Good statistical practice includes communicating what the analysis **cannot** conclude.

## 32. Final Statistical Report

A concise final report can follow this structure:

### Research Objective
What questions were investigated?

### Data
What observations and variables were analyzed?

### Exploratory Findings
What did the distributions and visualizations reveal?

### Inferential Findings
What evidence was found for group differences or associations?

### Modeling Results
Which variables were associated with performance and completion?

### Uncertainty
What do confidence intervals and bootstrap results tell us?

### Practical Interpretation
Which findings appear meaningful in context?

### Limitations
What conclusions should we avoid?

### Conclusion
What is the overall answer to the original research problem?

## 33. Exercises — Your Turn

1. Create one new research question using this dataset.
2. Identify the relevant outcome and explanatory variables.
3. Classify their variable types.
4. Choose an appropriate statistical method.
5. State $H_0$ and $H_1$ if hypothesis testing is appropriate.
6. Create at least one visualization.
7. Perform the analysis in Python.
8. Interpret both statistical and practical significance.
9. Explain at least one limitation.
10. Write a short conclusion in plain language.

## 34. Summary

Chapter 23 integrated the statistical workflow developed throughout this repository:

$$
\text{Question}
\rightarrow
\text{Data}
\rightarrow
\text{Exploration}
\rightarrow
\text{Inference}
\rightarrow
\text{Modeling}
\rightarrow
\text{Interpretation}
$$

The central lesson is:

> **Statistical analysis is not a collection of tests. It is a process of turning questions into evidence.**

A strong analysis does not use the largest possible number of methods. It uses methods that are appropriate for the research question and communicates the resulting uncertainty and limitations clearly.

## 35. Next Chapter

**Chapter 24 — Bridge to Multivariate Analytics**

So far, we have used multiple variables in regression and other statistical analyses.

The next question is:

> **What happens when the relationships among many variables become the main object of analysis?**

Chapter 24 will introduce the motivation for multivariate analysis and build a bridge toward methods such as Principal Component Analysis and clustering, which will be developed further in the **Multivariate-Analytics** repository.